# Verify Final Sample Size (n = 3,371)

This notebook independently re-derives the final analytic sample size, step by step, from the raw DHS 2022 IR file. Run each cell in order and check the printed shape after each filtering step.

**Update the file path in Cell 1 to match your raw DHS file location.**

In [ ]:
import pandas as pd

# UPDATE THIS PATH to your actual raw DHS IR file location
RAW_PATH = '../data/raw/NPIR8XFL.DTA'  # <-- change filename if different

df = pd.read_stata(RAW_PATH, convert_categoricals=False)
print("Step 0 - Full raw dataset shape:", df.shape)

In [ ]:
# Step 1: Check v025 (place of residence) distribution before filtering
print("v025 value counts:")
print(df['v025'].value_counts(dropna=False))
print("\nv025 percentage:")
print(df['v025'].value_counts(normalize=True, dropna=False) * 100)

In [ ]:
# Step 2: Filter to rural women only (v025 == 2)
rural_df = df[df['v025'] == 2].copy()
print("Step 2 - Rural-only shape:", rural_df.shape)

In [ ]:
# Step 3: Check age range sanity (should be 15-49, DHS women's questionnaire scope)
print("Age (v012) range:", rural_df['v012'].min(), "-", rural_df['v012'].max())
print("Rural women aged 15-49:", rural_df.shape[0])

In [ ]:
# Step 4: Check the anemia/hemoglobin target variable's missingness
# v457 = anemia level (categorical); v456 = hemoglobin, altitude+smoking adjusted (continuous)
print("v457 (anemia level) value counts, including missing:")
print(rural_df['v457'].value_counts(dropna=False))
print("\nMissing (untested) percentage:", rural_df['v457'].isnull().mean() * 100, "%")

In [ ]:
# Step 5: Derive binary anemia target and drop untested women
# v457 codes: 1=severe, 2=moderate, 3=mild anemia, 4=not anemic (verify against your DHS recode manual)
rural_df['anemia_binary'] = rural_df['v457'].map({1: 1, 2: 1, 3: 1, 4: 0})

before = rural_df.shape[0]
final_df = rural_df[rural_df['anemia_binary'].notna()].copy()
after = final_df.shape[0]

print(f"Before dropping untested women: {before}")
print(f"After dropping untested women: {after}")
print(f"Dropped: {before - after} ({(before - after) / before * 100:.1f}%)")

In [ ]:
# Step 6: Final check - does this match n = 3,371?
print("FINAL SAMPLE SIZE:", final_df.shape[0])
print("Expected: 3371")
print("Match:", final_df.shape[0] == 3371)

print("\nFinal class balance:")
print(final_df['anemia_binary'].value_counts(normalize=True) * 100)

In [ ]:
# Step 7 (optional): If the count doesn't match, check for additional filtering
# that may have been applied later (e.g. dropping rows with missing predictor values
# in your 12 selected features). Uncomment and adjust to test:

# selected_features = ['v190a', 'v106', 'v133', 'm14_1', 'v116', 'v113',
#                       'v012', 'v013', 'v024', 'secoreg', 'v040']
# check_df = final_df.dropna(subset=selected_features)
# print("Shape after also requiring complete predictor data:", check_df.shape)